# Day 07 — OOP Part II: Access Control · Static · Class Methods · Inheritance · Composition
**Module 01: Python + Async + FastAPI for LLM Engineering**  
Vidya Sankalp | Applied GenAI Engineering

> **What you'll learn:** `_protected` · `__private` · `@staticmethod` · `@classmethod` · inheritance · `super()` · method override · `isinstance()` · composition

> **Building on Day 06:** You know how to define a class, write `__init__`, add methods, and use `@property`. Today you learn how to control access to data, add class-level behaviour, and build class hierarchies.


---


## 1 — Private and Protected Attributes

Python does **not** enforce access control the way Java or C# does. Instead it uses **naming conventions**.

| Convention | Syntax | Meaning | Java / C# |
|-----------|--------|---------|----------|
| Public | `self.name` | Anyone can read and write | `public` |
| Protected | `self._name` | Single underscore — *"internal use, don't touch from outside"* | `protected` |
| Private | `self.__name` | Double underscore — Python renames it to prevent easy access | `private` |

> **Python's philosophy:** *"We are all consenting adults."*  
> Nothing is truly hidden — but the convention tells other developers: *"don't use this directly."*


In [1]:
class ProductReview:

    def __init__(self, review_id, rating, title):
        self.review_id  = review_id    # public   — read/write freely
        self._score     = rating * 20  # protected — internal calculation
        self.__raw_text = title        # private   — Python renames this

    def get_title(self):
        # The class itself can access __raw_text normally
        return self.__raw_text.strip().title()


r = ProductReview(5001, 4, '  really good monitor  ')

# Public — accessible anywhere
print('public  :', r.review_id)

# Protected — accessible but the _ is a signal: 'internal use'
print('protected:', r._score)

# Private — Python renames __raw_text to _ProductReview__raw_text
# r.__raw_text              # AttributeError
print('private (renamed):', r._ProductReview__raw_text)  # still accessible if you know the trick
print('via method       :', r.get_title())


public  : 5001
protected: 80
private (renamed):   really good monitor  
via method       : Really Good Monitor


In [5]:
# Practical pattern — use _ for internal state you want to protect
class LLMConfig:

    def __init__(self, model, api_key):
        self.model   = model          # public — fine to read
        self.__api_key = api_key      # private — never expose in logs or repr


    def get_api_key(self, persona):
        if persona == 'admin':
            return self.__api_key
        else:
            return '*****'

    def set_api_key(self, persona, value):
        if persona == 'admin':
            self.__api_key = value
        else:
            print("You are not allowed to change the API key")



    def __repr__(self):
        # Notice: __api_key is NOT shown — never log secrets
        return f'LLMConfig(model={self.model!r}, api_key="***")'

    def has_key(self):
        return bool(self.__api_key)


# cfg = LLMConfig('gpt-4o', 'sk-abc123')
# print(cfg)              # api_key hidden in repr
# print(cfg.has_key())   # True — method can use it internally
# # print(cfg.__api_key)  # AttributeError — good!
# cfg.get_api_key('developer')
# cfg.set_api_key('admin', 'sk-abc234')
# print(cfg.get_api_key('admin'))




LLMConfig(model='gpt-4o', api_key="***")
True
sk-abc234


---


## 2 — `@staticmethod` — Belongs to the Class, Needs Nothing From It

A static method belongs to the class but does **not** receive `self` or `cls`.  
It cannot access instance data or class data — it is just a regular function that lives inside the class namespace.

```python
@staticmethod
def my_util(value):   # no self, no cls
    return value * 2
```

| | `self` | `cls` | Can access instance data? | Can access class data? |
|-|--------|-------|--------------------------|------------------------|
| instance method | ✅ | ❌ | ✅ | ✅ |
| `@classmethod` | ❌ | ✅ | ❌ | ✅ |
| `@staticmethod` | ❌ | ❌ | ❌ | ❌ |

**When to use `@staticmethod`:** utility or validation logic that is logically related to the class but does not need any object or class data.

Java/C#: `public static` method.


In [25]:
a = 1
b = 2
if isinstance(a,int) and isinstance(b,int):
    c = a + b
    print(c)

3


In [27]:
class ProductReview:

    def __init__(self, review_id, rating, title, verified=False):
        self.review_id = review_id
        self.rating    = rating
        self.title     = title.strip()
        self.verified  = verified


    def rating_regular_method(self):
        return isinstance(self.rating, int) and 1 <= self.rating <= 5

    @staticmethod
    def is_valid_rating(rating):
        """
        Check if a rating value is valid.
        @staticmethod — no self or cls — just a utility function.
        Called as: ProductReview.is_valid_rating(4)
        """
        return isinstance(rating, int) and 1 <= rating <= 5

    @staticmethod
    def rating_to_stars(rating):
        """Convert a number to a star string. Useful outside of any instance."""
        return '★' * rating + '☆' * (5 - rating)

    @property
    def stars(self):
        # Instance method reuses the static method
        return ProductReview.rating_to_stars(self.rating)

    def __repr__(self):
        return f'ProductReview(id={self.review_id}, rating={self.rating})'


# Call on the class — no instance needed
print(ProductReview.is_valid_rating(4))   # True
print(ProductReview.is_valid_rating(6))   # False
print(ProductReview.rating_to_stars(3))   # ★★★☆☆

# Can also call on an instance — works but unconventional
r = ProductReview(5001, 4, 'Great monitor')
print(r.is_valid_rating(2))   # True — works, but prefer the class call above

print(isinstance(r, ProductReview))


True
False
★★★☆☆
True
True


---


## 3 — `@classmethod` — Alternative Constructors

A class method receives the **class itself** as its first argument (`cls` instead of `self`).  
The most common use is as an **alternative constructor** — a second way to create an object.

```python
@classmethod
def from_something(cls, data):
    return cls(...)   # cls(...) is the same as ProductReview(...)
```

Java/C#: static factory method — `ProductReview.fromCsvRow(row)`

**Why `cls` instead of the class name directly?**  
If a subclass calls `from_csv_row()`, `cls` will be the subclass — not `ProductReview`. This makes the method work correctly for inheritance.


In [28]:
class ProductReview:

    def __init__(self, review_id, rating, title, verified=False):
        self.review_id = review_id
        self.rating    = rating
        self.title     = title.strip()
        self.verified  = verified

    @classmethod
    def from_csv_row(cls, row):
        """
        Alternative constructor — create a ProductReview from a csv.DictReader row.
        CSV gives all values as strings — we convert types here.
        cls = ProductReview (or a subclass if called on one).
        """
        return cls(
            review_id = int(row['review_id']),
            rating    = int(row['rating']),
            title     = row['review_title'],
            verified  = row.get('verified_purchase', 'No') == 'Yes',
        )

    @classmethod
    def from_dict(cls, data):
        """Alternative constructor — create from a plain Python dict."""
        return cls(
            review_id = data['review_id'],
            rating    = data['rating'],
            title     = data['title'],
            verified  = data.get('verified', False),
        )

    @staticmethod
    def is_valid_rating(rating):
        return isinstance(rating, int) and 1 <= rating <= 5

    @property
    def stars(self):
        return '★' * self.rating + '☆' * (5 - self.rating)

    def __repr__(self):
        return f'ProductReview(id={self.review_id}, rating={self.rating}, verified={self.verified})'


# Standard constructor
r1 = ProductReview(5001, 4, 'Great monitor')

# From a CSV row (all strings — from_csv_row converts types)
csv_row = {'review_id': '5002', 'rating': '5', 'review_title': 'Excellent!', 'verified_purchase': 'Yes'}
r2 = ProductReview.from_csv_row(csv_row)

# From a plain dict
r3 = ProductReview.from_dict({'review_id': 5003, 'rating': 2, 'title': 'Disappointed'})

print(r1)
print(r2)
print(r3)


ProductReview(id=5001, rating=4, verified=False)
ProductReview(id=5002, rating=5, verified=True)
ProductReview(id=5003, rating=2, verified=False)


In [29]:
# Practical example — load all reviews from simulated CSV data
import io, csv

sample_csv = """review_id,rating,review_title,verified_purchase
5001,5,Excellent product,Yes
5002,2,Disappointed,No
5003,4,Really good,Yes
"""

reader = csv.DictReader(io.StringIO(sample_csv))

# from_csv_row handles all the type conversion cleanly
reviews = [ProductReview.from_csv_row(row) for row in reader]

for r in reviews:
    print(f'  {r.stars}  {r}')


  ★★★★★  ProductReview(id=5001, rating=5, verified=True)
  ★★☆☆☆  ProductReview(id=5002, rating=2, verified=False)
  ★★★★☆  ProductReview(id=5003, rating=4, verified=True)


---


## 4 — Inheritance — One Class Building on Another

Inheritance lets a **child class** reuse all the code from a **parent class** and add or change behaviour.

```python
class Parent:
    ...

class Child(Parent):   # Child inherits everything from Parent
    ...
```

| Term | Meaning | Java/C# |
|------|---------|--------|
| Parent / Base class | The class being inherited from | `extends` / `: BaseClass` |
| Child / Subclass | The class that inherits | `extends` / `: BaseClass` |
| `super()` | Access the parent class | `super()` / `base` |
| Override | Replace the parent's method with a new one | `@Override` / `override` |

**When to use inheritance:** when the child IS A type of the parent. `OrderAgent` IS AN `LLMAgent`.


In [3]:
# ── Base class — shared by all LLM agents ────────────────────
class LLMAgent:

    def __init__(self, name, model='gpt-4o'):
        self.name  = name
        self.model = model
        self._call_count = 0    # protected — track how many calls made


    def call(self, prompt):
        """Send a prompt and return a simulated response."""
        self._call_count += 1
        return f'[{self.name}] Response to: {prompt[:40]}'

    def status(self):
        return f'{self.name} | model={self.model} | calls={self._call_count}'

    def __repr__(self):
        return f'LLMAgent(name={self.name!r}, model={self.model!r})'


# Create and use the base class
agent = LLMAgent('base_agent')
print(agent.call('What is the weather?'))
print(agent.status())


[base_agent] Response to: What is the weather?
base_agent | model=gpt-4o | calls=1


In [16]:
# ── Child class — inherits from LLMAgent ─────────────────────
class OrderAgent(LLMAgent):
    """
    Specialised agent for handling order tracking queries.
    Inherits call() and status() from LLMAgent.
    Adds: route(), handles order-specific logic.
    """

    def __init__(self, name , model='gpt-4o'):
        # super().__init__() calls the PARENT __init__
        # Java/C#: super(name, 'gpt-4o') / base(name, 'gpt-4o')
        super().__init__(name, model='gpt-4o')
        # LLMAgent.__init__(self, name,model)
        self._call_count = 1
        self.handled_orders = []   # extra state only OrderAgent has

    def route(self, query):
        """Decide if this agent should handle this query."""
        keywords = ['order', 'track', 'where', 'delivery', 'shipped']
        return any(kw in query.lower() for kw in keywords)


    # Override — replace parent's call() with a specialised version
    def call(self, prompt):
        self._call_count += 1   # still track calls
        order_id = 'ORD-3042'   # in production: extracted from prompt
        self.handled_orders.append(order_id)
        return f'[OrderAgent] Order {order_id} is In Transit, arriving Friday.'

    def call_parent(self, prompt):
        super().call(prompt)

    def __repr__(self):
        return f'OrderAgent(name={self.name!r}, handled={len(self.handled_orders)})'


order_agent = OrderAgent('order_agent')

# Methods inherited from LLMAgent
print(order_agent.status())      # inherited

# Overridden method
print(order_agent.call('Where is my order?'))   # uses OrderAgent.call()
print(order_agent.status())      # call_count updated

# Extra method only on OrderAgent
print(order_agent.route('Where is my order?'))  # True
print(order_agent.route('Do you have discounts?'))  # False


order_agent | model=gpt-4o | calls=1
[OrderAgent] Order ORD-3042 is In Transit, arriving Friday.
order_agent | model=gpt-4o | calls=2
True
False


In [18]:
# ── Second child class ───────────────────────────────────────
class RefundAgent(LLMAgent):

    def __init__(self, name):
        super().__init__(name, model='gpt-4o')

    def route(self, query):
        keywords = ['refund', 'cancel', 'return', 'money back']
        return any(kw in query.lower() for kw in keywords)

    def call(self, prompt):
        self._call_count += 1
        return f'[RefundAgent] Refund initiated. Allow 3-5 business days.'

    def __repr__(self):
        return f'RefundAgent(name={self.name!r})'


# Both agents share the same parent but behave differently
agents = [OrderAgent('order_agent'), RefundAgent('refund_agent')]

query = 'I want to cancel my order'

# ITERATION TRACE:
# agent=OrderAgent  → route('I want to cancel...') → 'order' not in q, 'cancel' not in keywords → False
# agent=RefundAgent → route('I want to cancel...') → 'cancel' in keywords → True → call()
no_agent = 0
for agent in agents:
    if agent.route(query):
        print(f'Routed to {agent.name}')
        print(agent.call(query))
        no_agent += 1

if no_agent == 0:
    print('No agent called')



Routed to order_agent
[OrderAgent] Order ORD-3042 is In Transit, arriving Friday.
Routed to refund_agent
[RefundAgent] Refund initiated. Allow 3-5 business days.


### What `super()` does

```python
class OrderAgent(LLMAgent):
    def __init__(self, name):
        super().__init__(name, model='gpt-4o')
#       ↑
#       Calls LLMAgent.__init__(self, name, model='gpt-4o')
#       Sets up self.name, self.model, self._call_count
#       WITHOUT super() those attributes would never be created
```

> **Rule:** If a child class defines `__init__`, always call `super().__init__(...)` first — otherwise the parent's setup code never runs.


---


## 5 — `isinstance()` — Check What Type an Object Is

| Function | Returns True when | Java/C# |
|----------|------------------|---------|
| `isinstance(obj, MyClass)` | obj is MyClass **or any subclass** | `instanceof` / `is` |
| `type(obj) == MyClass` | obj is **exactly** MyClass — subclasses return False | `obj.GetType() == typeof(MyClass)` |

**Always prefer `isinstance()`** — it handles inheritance correctly.


In [19]:
order_agent  = OrderAgent('order_agent')
refund_agent = RefundAgent('refund_agent')
base_agent   = LLMAgent('base')

# isinstance() — True for the class AND all subclasses
print(isinstance(order_agent,  LLMAgent))     # True  — OrderAgent IS AN LLMAgent
print(isinstance(order_agent,  OrderAgent))   # True
print(isinstance(order_agent,  RefundAgent))  # False — different subclass
print(isinstance(base_agent,   LLMAgent))     # True

print()
# type() — strict exact match only
print(type(order_agent) == LLMAgent)    # False — it's OrderAgent, not LLMAgent
print(type(order_agent) == OrderAgent)  # True


True
True
False
True

False
True


In [20]:
# Practical use — a function that handles any agent type
def dispatch(query, agents):
    """Route a query to the first agent that can handle it."""
    for agent in agents:
        # isinstance() ensures we only call .route() on agents that have it
        if isinstance(agent, LLMAgent) and agent.route(query):
            return agent.call(query)
    return 'No agent could handle this query'

agents = [OrderAgent('order_agent'), RefundAgent('refund_agent')]

print(dispatch('Where is my order #3042?', agents))
print(dispatch('I want a refund please',   agents))
print(dispatch('Tell me a joke',           agents))


[OrderAgent] Order ORD-3042 is In Transit, arriving Friday.
[RefundAgent] Refund initiated. Allow 3-5 business days.
No agent could handle this query


---


## 6 — Composition — An Object That Contains Another Object

Composition means one class **uses** another class by storing it as an instance variable.

| | Inheritance | Composition |
|-|-------------|-------------|
| Relationship | IS-A (`OrderAgent` IS AN `LLMAgent`) | HAS-A (`ChatAgent` HAS A `ConversationHistory`) |
| Syntax | `class Child(Parent):` | `self.history = ConversationHistory()` |
| Java/C# | `extends` / `:` | field of another type |
| Use when | Child is a specialised version of parent | You want to reuse behaviour without being a subtype |

> **Rule of thumb:** prefer composition over inheritance when in doubt.  
> Inheritance couples two classes tightly — composition keeps them independent.


In [5]:
# ── The contained class ──────────────────────────────────────
class ConversationHistory:
    """Stores and manages the message history for one conversation."""

    def __init__(self):
        self._messages = []   # protected — manage through methods

    def add(self, role, content):
        self._messages.append({'role': role, 'content': content})

    def get_all(self):
        return list(self._messages)   # return a copy, not the original

    def clear(self):
        self._messages = []

    @property
    def turn_count(self):
        return len(self._messages)

    def __repr__(self):
        return f'ConversationHistory(turns={self.turn_count})'


# ── The containing class ─────────────────────────────────────
class ChatAgent(LLMAgent):
    """
    A conversational agent that maintains message history.
    HAS-A ConversationHistory — does not inherit from it.
    """

    def __init__(self, name, model='gpt-4o'):
        super().__init__(name, model)
        self.history = ConversationHistory()   # HAS-A — composition

    def chat(self, user_message):
        """Add user message, call LLM, add response, return response."""
        self.history.add('user', user_message)
        response = self.call(user_message)     # inherited from LLMAgent
        self.history.add('assistant', response)
        return response

    def __repr__(self):
        return f'ChatAgent(name={self.name!r}, history={self.history})'


agent = ChatAgent('support_agent')

print(agent.chat('Where is my order #3042?'))
print(agent.chat('Can I change the delivery address?'))
print()
print(f'Turn count: {agent.history.turn_count}')
print(f'Full history:')
for msg in agent.history.get_all():
    print(f'  {msg["role"]:12s}: {msg["content"][:50]}')


[support_agent] Response to: Where is my order #3042?
[support_agent] Response to: Can I change the delivery address?

Turn count: 4
Full history:
  user        : Where is my order #3042?
  assistant   : [support_agent] Response to: Where is my order #30
  user        : Can I change the delivery address?
  assistant   : [support_agent] Response to: Can I change the deli


In [ ]:
# Composition advantage — ConversationHistory can be reused anywhere
# without being tied to ChatAgent through inheritance

# Two separate agents — each has its OWN ConversationHistory
agent_a = ChatAgent('agent_a')
agent_b = ChatAgent('agent_b')

agent_a.chat('Hello from customer A')
agent_a.chat('Where is order #3042?')
agent_b.chat('Hello from customer B')

# Each agent's history is independent
print('agent_a turns:', agent_a.history.turn_count)   # 4 (2 user + 2 assistant)
print('agent_b turns:', agent_b.history.turn_count)   # 2


---


## 7 — Putting It All Together

A quick view of how all today's concepts appear in the same class:


In [ ]:
class LLMAgent:
    """Base class — all agents inherit from this."""

    # ── public, protected, private ───────────────
    def __init__(self, name, model='gpt-4o'):
        self.name        = name           # public
        self.model       = model          # public
        self._call_count = 0              # protected
        self.__secret    = 'internal'     # private

    # ── instance method ──────────────────────────
    def call(self, prompt):
        self._call_count += 1
        return f'[{self.name}] {prompt[:40]}'

    # ── @staticmethod ────────────────────────────
    @staticmethod
    def is_valid_model(model):
        """No self or cls — pure utility."""
        return model in {'gpt-4o', 'gpt-3.5-turbo', 'claude-3-5-sonnet'}

    # ── @classmethod ─────────────────────────────
    @classmethod
    def from_config(cls, config):
        """Alternative constructor from a config dict."""
        return cls(config['name'], config.get('model', 'gpt-4o'))

    # ── @property ────────────────────────────────
    @property
    def call_count(self):
        return self._call_count

    def __repr__(self):
        return f'LLMAgent(name={self.name!r}, model={self.model!r}, calls={self._call_count})'


# Static method — no instance needed
print(LLMAgent.is_valid_model('gpt-4o'))   # True

# Class method — alternative constructor
agent = LLMAgent.from_config({'name': 'support_agent', 'model': 'gpt-4o'})
print(agent)

# Regular use
agent.call('Where is order #3042?')
print(f'Calls made: {agent.call_count}')   # @property


---


## Summary — Day 07

| Concept | Python | Java / C# |
|---------|--------|-----------|
| Public | `self.name` | `public` |
| Protected | `self._name` — convention only | `protected` |
| Private | `self.__name` — name-mangled | `private` |
| Static method | `@staticmethod` — no self or cls | `static` method |
| Class method | `@classmethod` — receives `cls` | static factory method |
| Inherit | `class Child(Parent):` | `extends` / `: BaseClass` |
| Call parent | `super().__init__(...)` | `super()` / `base(...)` |
| Override | define same method name in child | `@Override` / `override` |
| Type check | `isinstance(obj, MyClass)` | `instanceof` / `is` |
| Exact type | `type(obj) == MyClass` | `obj.GetType() == typeof(MyClass)` |
| Composition | `self.history = ConversationHistory()` | field of another type |

**Key rules:**
- `_name` = "I'm internal — don't use me directly" (not enforced)
- `__name` = Python renames to `_ClassName__name` — harder to access accidentally
- `@staticmethod` — no instance needed, just a utility that belongs with the class
- `@classmethod` — use for alternative constructors (`from_csv_row`, `from_config`)
- Always call `super().__init__()` in a child class `__init__`
- Prefer `isinstance()` over `type()` — handles subclasses correctly
- IS-A → inheritance · HAS-A → composition

**Next:** Day 08 — Async Python  
`async def` · `await` · `asyncio.gather()` · calling the LLM API asynchronously


In [10]:
class ProductReview:

    def __init__(self, review_id, rating, title, verified=False):
        self.review_id = review_id
        self.rating    = rating
        self.title     = title.strip()
        self.verified  = verified

    @property
    def stars(self):
        """Rating as filled/empty stars. Access as r.stars (no parentheses)."""
        return '★' * self.rating + '☆' * (5 - self.rating)

    @property
    def is_positive(self):
        """True if rating is 4 or above."""
        return self.rating >= 4

    @property
    def summary(self):
        """One-line summary — useful for logging and prompt injection."""
        badge = '[Verified] ' if self.verified else ''
        return f'{self.stars} {badge}{self.title}'

    def to_context(self):
        return (
            f'<review>\n'
            f'  Rating  : {self.stars} ({self.rating}/5)\n'
            f'  Title   : {self.title}\n'
            f'</review>'
        )

    def __repr__(self):
        return f'ProductReview(review_id={self.review_id}, title="{self.title}",rating={self.rating}, verified={self.verified})'

In [20]:
reviews = []
with open('reviews.txt','r') as f:
    lines = f.readlines()
    for line in lines:
        reviews.append(eval(line))

In [13]:
for review in reviews:
    print(type(review))

<class '__main__.ProductReview'>
<class '__main__.ProductReview'>
<class '__main__.ProductReview'>
<class '__main__.ProductReview'>
<class '__main__.ProductReview'>


SyntaxError: invalid syntax (<string>, line 1)

In [22]:
class ProductReview:
    def __init__(self, review_id):
        self.review_id = review_id
        print(self)


    def __repr__(self):
        return f'ProductReview(review_id={self.review_id})'

r = ProductReview(5001)

ProductReview(review_id=5001)


In [1]:

aDict = {
    "tmp":0.1,
    max_token:1024
}
print(aDict)

NameError: name 'tmp' is not defined